Imports and Installs

In [2]:
# Import drive for files
from google.colab import drive
drive.mount('/content/drive')

# Installation
!pip -q install pymupdf pdfplumber pandas pylatexenc

# Packages
import re, json
from pathlib import Path
import fitz
import pdfplumber
import pandas as pd
from pylatexenc.latexencode import unicode_to_latex

pdf_directory = Path("/content/drive/MyDrive/Math Olympiad Competition/EC-Olympiad-pdfs")
out_dir = Path("/content/drive/MyDrive/Math Olympiad Competition/EC-Olympiad-output")
out_dir.mkdir(parents=True, exist_ok=True)

csv_out = out_dir / "ec_qa.csv"
jsonl_out = out_dir / "ec_qa.jsonl"


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 95.8 MB/s eta 0:00:00


In [29]:

SOL_RE = re.compile(
    r"(?mi)^\s*(?:"
    r"Solution|Proof|Answer|Discussion|Commentary|"
    r"Solution\s*\([A-E]\)|"
    r"Sketch|"
    r"Approach"
    r")\s*[:\.]?\s*$"
)

PROBSTAT_RE = re.compile(r"(?mi)^\s*Problem\s+statement\s*$")

def split_problem_solution(chunk: str):
    # Prefer splitting on "Problem statement" first (common in solution notes)
    pm = PROBSTAT_RE.search(chunk)
    if pm:
        # After "Problem statement", the rest often contains both problem and solution;
        # so we then try to find "Solution" inside that remainder.
        after = chunk[pm.end():].strip()
        sm = SOL_RE.search(after)
        if sm:
            prob_raw = after[:sm.start()].strip()
            sol_raw  = after[sm.end():].strip()
            return prob_raw, sol_raw, None


    sm = SOL_RE.search(chunk)
    if not sm:
        return None
    return chunk[:sm.start()].strip(), chunk[sm.end():].strip(), None


In [30]:
def id_function(i: int) -> str:
    # there are 26^3*100 = possible matches
    numbers = f"{i % 1000:03d}"
    x = i // 1000
    letters = "".join(chr(ord("a") + ((x // (26**j)) % 26)) for j in reversed(range(3)))
    return numbers + letters

# Latex conversion rules
latex_conversion = {
    "≤": r"\le", "≥": r"\ge", "≠": r"\ne", "≈": r"\approx",
    "×": r"\times", "·": r"\cdot", "÷": r"\div",
    "π": r"\pi", "θ": r"\theta", "∞": r"\infty",
    "ℝ": r"\mathbb{R}", "ℤ": r"\mathbb{Z}", "ℚ": r"\mathbb{Q}", "ℕ": r"\mathbb{N}",
    "∠": r"\angle", "…": r"\ldots",
}
# Suggested cleanups for latex
def cleanup_latex(s: str) -> str:
    s = re.sub(r"\\ensuremath\{([^{}]+)\}", r"\1", s)
    s = s.replace(r"{\textrightarrow}", r"\to").replace(r"{\textdegree}", r"^\circ")
    s = s.replace(r"{\textemdash}", "---").replace(r"{\textendash}", "--")
    s = s.replace(r"{\textquotedblleft}", '"').replace(r"{\textquotedblright}", '"')
    s = s.replace(r"{\textquoteleft}", "'").replace(r"{\textquoteright}", "'")
    s = s.replace(r"\ensuremath{\sqrt{}}", r"\sqrt")
    s = re.sub(r"(\\angle)([A-Z])", r"\1 \2", s)
    s = re.sub(r"(\\sqrt)\{\}\s*(\d+)", r"\1{\2}", s)
    s = re.sub(r"(\\sqrt)\{\}\s*([a-zA-Z])", r"\1{\2}", s)
    s = re.sub(r"\\sqrt(\d+)", r"\\sqrt{\1}", s)
    return s

# Clean text by applying two previous functions
def convert_text(t: str) -> str:
    for x, v in latex_conversion.items():
        t = t.replace(x, v)
    t = unicode_to_latex(t, non_ascii_only=True)
    t = cleanup_latex(t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

In [22]:
# In here extract text with pymudpdf
def extract_with_pymupdf(pdf_path: Path) -> str:
    pages_text = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            blocks = page.get_text("blocks")
            blocks = sorted(blocks, key=lambda b: (b[1], b[0]))
            pages_text.append("\n".join(b[4].strip() for b in blocks if b[4].strip()))
    return "\n\n".join(pages_text)

# In here extract text with pdfplumber
def extract_with_pdfplumber(pdf_path: Path) -> str:
    pages = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            pages.append(page.extract_text(layout=True) or "")
    return "\n\n".join(pages)
# Try PyMuPDF first if extraction is very short or fails fall back to pdfplumber
def extract_text_from_pdf(pdf_path: Path) -> str:
    try:
        txt = extract_with_pymupdf(pdf_path)
        if len(txt.strip()) > 200:
            return txt
    except Exception as e:
        print(f"[warn] PyMuPDF failed on {pdf_path.name}: {e}")
    try:
        return extract_with_pdfplumber(pdf_path)
    except Exception as e:
        print(f"[error] pdfplumber failed on {pdf_path.name}: {e}")
        return ""

Build CSV ND jsonl

In [33]:
# ---- BUILD ROWS FROM PDFs ----

pdf_files = sorted(pdf_directory.rglob("*.pdf"))
print("PDFs found:", len(pdf_files))
print("First few:", [p.name for p in pdf_files[:5]])

rows = []

SECTION_RE = re.compile(
    r"(?m)^\s*(?:§|\\S)\s*\d+(?:\.\d+)?\s+.*?/\s*(\d{1,2})\b"
)


def split_solution_sections(txt: str):
    matches = list(SECTION_RE.finditer(txt))
    out = []
    for i, m in enumerate(matches):
        qnum = int(m.group(1))
        start = m.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(txt)
        out.append((qnum, txt[start:end].strip()))
    return out

# Matches starts of solution-ish paragraphs in Evan Chen notes.
# Works whether the line begins with ¶ or \P or nothing.
SPLIT_MARK_RE = re.compile(
    r"(?mi)^\s*(?:¶|\\P)?\s*(Answer|Solution|Proof|Construction|Discussion|Lemma|Claim|Step)\b"
)

PROBSTAT_RE = re.compile(r"(?mi)^\s*Problem\s+statement\s*$")

def split_problem_solution_from_section_raw(section_text: str):
    pm = PROBSTAT_RE.search(section_text)
    if not pm:
        return None

    after = section_text[pm.end():].strip()
    if not after:
        return None

    sm = SPLIT_MARK_RE.search(after)
    if not sm:
        # no clear marker; skip rather than guessing (you can loosen later)
        return None

    prob = after[:sm.start()].strip()
    sol  = after[sm.start():].strip()
    if not prob or not sol:
        return None
    return prob, sol


for pdf_path in pdf_files:
  txt_raw = extract_text_from_pdf(pdf_path)
  if not txt_raw.strip():
      continue

  # IMPORTANT: split sections BEFORE convert_text()
  sections = split_solution_sections(txt_raw)

  for qnum, section_raw in sections:
    ps = split_problem_solution_from_section_raw(section_raw)
    if not ps:
        continue
    problem_raw, solution_raw = ps

    rows.append({
        "pdf": pdf_path.name,
        "qnum": qnum,
        "problem": convert_text(problem_raw),
        "solution": convert_text(solution_raw),
    })

  print(pdf_path.name, "sections:", len(sections), "has Problem statement:", found_ps, "has Solution:", found_sol)

print("Rows collected:", len(rows))


PDFs found: 85
First few: ['EGMO-2012-notes.pdf', 'EGMO-2013-notes.pdf', 'EGMO-2014-notes.pdf', 'EGMO-2015-notes.pdf', 'EGMO-2016-notes.pdf']
EGMO-2012-notes.pdf sections: 8 has Problem statement: 6 has Solution: 0
EGMO-2013-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


EGMO-2014-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
EGMO-2015-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
EGMO-2016-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
EGMO-2017-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


EGMO-2018-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
EGMO-2019-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
EGMO-2020-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
EGMO-2021-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


IMO-1997-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
IMO-1998-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
IMO-1999-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


IMO-2000-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
IMO-2001-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
IMO-2002-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
IMO-2003-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


...

USAMO-2016-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
USAMO-2017-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
USAMO-2018-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


USAMO-2019-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
USAMO-2020-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


USAMO-2021-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
USAMO-2022-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


USAMO-2023-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0


USAMO-2024-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
USAMO-2025-notes.pdf sections: 6 has Problem statement: 6 has Solution: 0
Rows collected: 279


In [34]:
# ---- OUTPUT PATHS ----
out_dir = Path("/content/drive/MyDrive/Math Olympiad Competition/EC-Olympiad-output")
out_dir.mkdir(parents=True, exist_ok=True)

csv_out  = out_dir / "ec_qa.csv"
jsonl_out = out_dir / "ec_qa.jsonl"

# ---- SYSTEM PROMPT FOR CHAT FORMAT ----
SYSTEM_PROMPT = (
    "You are an expert math olympiad tutor. Solve the problem carefully and clearly, "
    "showing your reasoning and final answer."
)

print("Rows collected:", len(rows))
if len(rows) == 0:
    raise ValueError("No parsed rows found. Check your splitting regex / extraction output.")

# ---- CSV ----
df = pd.DataFrame(rows)

# Optional: enforce column order if you want consistency
preferred_cols = ["pdf", "qnum", "problem", "solution"]
cols = [c for c in preferred_cols if c in df.columns] + [c for c in df.columns if c not in preferred_cols]
df = df[cols]

df.to_csv(csv_out, index=False)
print("Wrote CSV:", csv_out)

# ---- JSONL (chat fine-tune style) ----
# Create stable IDs (pdf + qnum); you can also use uuid if you prefer
def make_id(pdf_name, qnum):
    base = f"{pdf_name}__{qnum}".replace(" ", "_")
    return base

with open(jsonl_out, "w", encoding="utf-8") as f:
    for r in rows:
        pdf_name = r.get("pdf", "unknown.pdf")
        qnum = r.get("qnum", "NA")

        prob = (r.get("problem") or "").strip()
        sol  = (r.get("solution") or "").strip()

        # Skip incomplete examples
        if not prob or not sol:
            continue

        obj = {
            "id": make_id(pdf_name, qnum),
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prob},
                {"role": "assistant", "content": sol},
            ],
            "meta": {
                "pdf": pdf_name,
                "qnum": qnum,
            }
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("Wrote JSONL:", jsonl_out)

# ---- quick peek ----
print(df.head(3))

Rows collected: 279
Wrote CSV: /content/drive/MyDrive/Math Olympiad Competition/EC-Olympiad-output/ec_qa.csv
Wrote JSONL: /content/drive/MyDrive/Math Olympiad Competition/EC-Olympiad-output/ec_qa.jsonl
                   pdf  qnum  \
0  EGMO-2012-notes.pdf     2   
1  EGMO-2012-notes.pdf     4   
2  EGMO-2012-notes.pdf     6   

                                             problem  \
0  Let n be a positive integer. Find the greatest...   
1  A set A of integers is called sum-full if A \s...   
2  There are infinitely many people registered on...   

                                            solution  
0  {\textparagraph} Answer.\nThe largest m is m(n...  
1  Claim ---\nLet N be an integer (possibly zero ...  
2  Claim --- Someone who is an n-best friend is a...  
